# Failure modes & repair loop (P0)

Demonstrates post-P0 behavior:
1. Validation ladder
2. Error-directed repair with early exit
3. Explicit extraction outcomes
4. Extraction gated on validation

No API keys required.


In [ ]:
import sys, os
from pathlib import Path
ROOT = Path.cwd()
if not (ROOT / 'tools').exists():
    ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))
from tools.linkml_tools import validate_linkml_schema
from tools.spires import run_spires_extraction
from tools.repair import repair_until_valid, fixture_regenerate, noop_regenerate
from tools.modes import ExtractionOutcome
print('ROOT =', ROOT)


## A — Invalid YAML


In [ ]:
INVALID_YAML = 'classes: [\n  this is not valid yaml\n'
report = validate_linkml_schema(INVALID_YAML)
print('valid:', report['valid'])
print('errors:', report['errors'])


In [ ]:
result = repair_until_valid(INVALID_YAML, fixture_regenerate, max_iterations=3)
print(result.stopped_reason, result.valid, result.schema_version)


## B — Broken OntoGPT conventions


In [ ]:
BROKEN = '''\nid: https://example.org/broken\nname: broken\nimports:\n  - linkml:types\nclasses:\n  Foo:\n    attributes:\n      bar:\n        range: string\n'''
print(validate_linkml_schema(BROKEN)['errors'])
fixed = repair_until_valid(BROKEN, fixture_regenerate, max_iterations=3)
print(fixed.stopped_reason, fixed.valid)


## C — Exhausted repair


In [ ]:
ex = repair_until_valid(BROKEN, noop_regenerate, max_iterations=2)
print(ex.stopped_reason, ex.valid)


## D — Extraction gated on invalid schema


In [ ]:
os.environ['AGENTIC_ONTOGPT_MODE'] = 'real'
blocked = run_spires_extraction('id: x\nname: x\nimports: []\nclasses: {}', 'text', require_valid_schema=True, validation_result={'valid': False, 'message': 'broken', 'errors': ['x']})
print(blocked['outcome'], blocked.get('error_type'))
assert blocked['outcome'] == ExtractionOutcome.REAL_EXTRACTION_FAILED.value


## E — Explicit simulation


In [ ]:
os.environ['AGENTIC_ONTOGPT_MODE'] = 'simulation'
sim = run_spires_extraction('id: x\nname: x\nimports: []\nclasses: {}', 'text', require_valid_schema=False)
print(sim['outcome'], sim.get('fixture'))
assert sim['outcome'] == ExtractionOutcome.SIMULATION_REQUESTED.value


## F — Real mode failure is not simulation


In [ ]:
os.environ['AGENTIC_ONTOGPT_MODE'] = 'real'
GOOD = '''id: https://w3id.org/ontogpt/demo\nname: demo\nimports:\n  - linkml:types\n  - core\nclasses:\n  ExtractionResult:\n    tree_root: true\n    attributes:\n      drugs:\n        range: Drug\n        multivalued: true\n  Drug:\n    is_a: NamedEntity'''
real = run_spires_extraction(GOOD, 'neutropenia after carboplatin', require_valid_schema=False)
print(real['outcome'], real['status'])


## G — Happy path


In [ ]:
GOOD2 = '''id: https://w3id.org/ontogpt/demo_clinical\nname: demo_clinical\nimports:\n  - linkml:types\n  - core\nclasses:\n  ExtractionResult:\n    tree_root: true\n    attributes:\n      medications:\n        range: Medication\n        multivalued: true\n  Medication:\n    is_a: NamedEntity\n    attributes:\n      id:\n        identifier: true\n        range: uriorcurie\n      label:\n        range: string'''
r = validate_linkml_schema(GOOD2)
print(r['valid'], [s['stage'] for s in r['stages'] if s['ok']])
